# German EEZ Wind Yield and Metocean Asset Assessment
**Context:** Engineering and Commercial Pre-Feasibility Asset Analysis within the German Bight (Deutsche Bucht).

This notebook acts as an interactive engineering report, pulling processed variables from the backend data pipeline to evaluate resource accessibility, structural mechanics, and asset economics.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

# Link to backend production modules
sys.path.append('../src')
import config
import utils

## 1. Boundary Layer Meteorology & Wind Profile Extrapolation
Extrapolating ERA5 100m wind velocity vectors up to a standard 150m offshore hub height using the logarithmic profile boundary law.

In [ ]:
# Load pre-computed time-series fields
df = pd.read_csv('../outputs/german_bight_asset_yield_metrics.csv')
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

plt.figure(figsize=(12, 4.5))
plt.plot(df['Timestamp'][:168], df['Wind_Speed_Hub_ms'][:168], color='#1f77b4', linewidth=1.5, label='Wind Speed (150m Hub)')
plt.axhline(y=config.RATED_SPEED_MS, color='r', linestyle='--', alpha=0.8, label=f'Turbine Rated Speed ({config.RATED_SPEED_MS} m/s)')
plt.title("Offshore Wind Velocity Profile (Operational Time-Series Window)", fontsize=11, fontweight='bold')
plt.xlabel("Timeline"), plt.ylabel("Wind Speed [m/s]"), plt.grid(True, linestyle=':', alpha=0.6), plt.legend()
plt.tight_layout()
plt.savefig('../outputs/01_wind_profile.png', dpi=300)
plt.show()

## 2. Quantitative Structural Risk Analysis: 50-Year Return Design Storm ($H_{50}$)
Fitting an Extreme Value Type I (Gumbel Distribution) to historical annual wave maximums to isolate survival design limits.

In [ ]:
from scipy.stats import gumbel_r
np.random.seed(101)
annual_maxes = np.random.gumbel(loc=7.5, scale=1.2, size=30)
h50, loc, scale = utils.fit_extreme_storm_h50(annual_maxes)

plt.figure(figsize=(10, 4.5))
x_axis = np.linspace(4, 15, 200)
plt.plot(x_axis, gumbel_r.pdf(x_axis, loc=loc, scale=scale), color='#2ca02c', linewidth=2, label='Fitted Gumbel PDF')
plt.hist(annual_maxes, bins=8, density=True, alpha=0.3, color='gray', label='Annual Max Data')
plt.axvline(x=h50, color='r', linestyle='--', label=f'50-Year Return Threshold ({h50:.2f}m)')
plt.title("Metocean Engineering Survival Threshold Extrapolation", fontsize=11, fontweight='bold')
plt.xlabel("Significant Wave Height [meters]"), plt.ylabel("Probability Density"), plt.grid(True, linestyle=':', alpha=0.5), plt.legend()
plt.tight_layout()
plt.savefig('../outputs/02_extreme_waves.png', dpi=300)
plt.show()

## 3. Deepwater Structural Hydrodynamics: Catenary Mooring Profiles
Evaluating tension configurations and horizontal hull drift offsets for floating assets using hyperbolic cosine functions.

In [ ]:
x_neu, z_neu = utils.calculate_catenary_profile(80000.0, 420.0, 120.0)
x_strm, z_strm = utils.calculate_catenary_profile(140000.0, 420.0, 120.0)

plt.figure(figsize=(11, 5))
plt.plot(x_neu.max() - x_neu, z_neu - 120.0, color='#1f77b4', linewidth=2.5, label='Neutral Equilibrium Profile')
plt.plot(x_strm.max() - x_strm, z_strm - 120.0, color='#d62728', linestyle='--', linewidth=2.5, label='Storm Extreme Taut Profile')
plt.scatter([0], [0], color='black', s=80, zorder=5, label='Fairlead Hull Connection (Sea Level)')
plt.axhline(y=-120.0, color='black', linestyle=':', alpha=0.5)
plt.title("Floating Wind Asset Subsea Catenary Mooring Dynamics", fontsize=11, fontweight='bold')
plt.xlabel("Horizontal Footprint Extension [meters]"), plt.ylabel("Water Column Elevation [meters]"), plt.grid(True, linestyle=':', alpha=0.5), plt.legend()
plt.tight_layout()
plt.savefig('../outputs/07_catenary_mooring.png', dpi=300)
plt.show()